In [135]:
import pandas as pd
import numpy as np

df = pd.read_csv("jopacc_system_month_november_2022_to_june_2026.csv")
df.head()


,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_usd_transaction_value,ach_eur_transaction_count,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level
0,2022-11,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
1,2022-11,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
2,2022-11,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
3,2022-12,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity
4,2022-12,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0,0,0.0,0.0,Low Activity


In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 50 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   report_month                     214 non-null    object 
 1   report_year                      214 non-null    int64  
 2   report_quarter                   214 non-null    object 
 3   system                           214 non-null    object 
 4   users_total                      214 non-null    int64  
 5   transactions_total               214 non-null    int64  
 6   transaction_value_jod            214 non-null    int64  
 7   user_jordanian_count             214 non-null    int64  
 8   user_non_jordanian_count         214 non-null    int64  
 9   user_male_count                  214 non-null    int64  
 10  user_female_count                214 non-null    int64  
 11  user_age_under_30_count          214 non-null    int64  
 12  user_age_31_50_count  

In [137]:
# temp features for both reg and class
numerical_features =  ["transactions_total",
                "returned_cheque_value_rate_pct",
                "returned_cheque_value_jod",
                "returned_cheque_count", 
                "returned_cheque_count_rate_pct",
                "off_us_money_transfer_count",
                "customer_legal_entity_count",
                "customer_individual_count",
                "ef_cash_payment_count",
                "ef_digital_payment_count",
                "users_total",
                "tx_cash_in_count",
                "tx_cash_out_count",
                "tx_purchases_count",
                "tx_money_transfers_count"
                ]


In [ ]:
# categorical_features= []

In [ ]:
#from sklearn.preprocessing import OneHotEncoder
#ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
#X_cat = ohe.fit_transform(df[categorical_features])


In [138]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(df[numerical_features])


In [139]:
import numpy as np

X = np.hstack([X_num_scaled])
X.shape

(214, 15)

In [140]:
import pickle
'''
cat_options = {
    col: list(cats)
    for col, cats in zip(categorical_features, ohe.categories_)
}

cat_options'''

'\ncat_options = {\n    col: list(cats)\n    for col, cats in zip(categorical_features, ohe.categories_)\n}\n\ncat_options'

In [141]:
import pickle

pickle.dump(scaler, open("scaler.pkl", "wb"))
# pickle.dump(ohe, open("onehot_encoder.pkl", "wb"))

# Correct: save numeric fields here
pickle.dump(numerical_features, open("num_cols.pkl", "wb"))

# Correct: this is empty because you chose no categories
# pickle.dump(categorical_features, open("cat_cols.pkl", "wb"))

In [142]:
# isolation forest 
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.03,  #   Expected fraction of anomalies in the data (3%)
    n_estimators=300,    #   Number of trees in the forest  (recommended ≥300)
    max_samples="auto",
    # ↑ Number of samples used to build each tree
    #   "auto" = min(256, n_samples)
    #   Smaller → more local sensitivity
    #   Larger → more global behavior
    max_features=1.0, #   Fraction of features randomly selected per tree  1.0 = use all features

    random_state=42
)

iso.fit(X)
df["score_iso"] = -iso.decision_function(X)
# decision_function:
#   positive → normal
#   negative → anomaly
# we flip sign so: higher score = more anomalous
df.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_eur_transaction_count,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,score_iso
0,2022-11,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0,0.0,0.0,Low Activity,-0.093743
1,2022-11,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0,0.0,0.0,Low Activity,-0.089450
2,2022-11,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0,0.0,0.0,Low Activity,-0.098492
3,2022-12,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0,0.0,0.0,Low Activity,-0.106223
4,2022-12,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0,0.0,0.0,Low Activity,-0.108297


In [143]:
threshold = np.percentile(df["score_iso"], 97)
df["anomaly_iso"] = (df["score_iso"] >= threshold).astype(int)
df.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_eur_transaction_value,ach_gbp_transaction_count,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,score_iso,anomaly_iso
0,2022-11,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0,0,0.0,0.0,Low Activity,-0.093743,0
1,2022-11,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0,0,0.0,0.0,Low Activity,-0.089450,0
2,2022-11,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0,0,0.0,0.0,Low Activity,-0.098492,0
3,2022-12,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0,0,0.0,0.0,Low Activity,-0.106223,0
4,2022-12,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0,0,0.0,0.0,Low Activity,-0.108297,0


In [144]:
df_synth = df.copy()
idx = df_synth.sample(frac=0.05, random_state=42).index
df_synth.loc[idx, "users_total"] *= 3
df_synth.loc[idx, "transactions_total"] *= 4          # meaningful perturbation
df_synth.loc[idx, "returned_cheque_count"] += 50       # inject a second signal
df_synth["is_synthetic"] = 0
df_synth.loc[idx, "is_synthetic"] = 1

In [145]:
from sklearn.metrics import roc_auc_score

# numeric features → scale using SAME scaler
X_num_synth = scaler.transform(df_synth[numerical_features])

# categorical → one-hot encode & ALIGN columns
# X_cat_synth = ohe.transform(df_synth[categorical_features])
X_synth = np.hstack([X_num_synth])


scores = -iso.decision_function(X_synth)

roc_auc_score(df_synth["is_synthetic"], scores)

#  AUC > 0.85 → strong detector
# 0.75–0.85  → good
# 0.65–0.75  → weak detector
#  ~0.50  →   broken evaluation

np.float64(0.8047469771607703)

In [146]:
import pickle

with open("isolation_forest_model.pkl", "wb") as f:
    pickle.dump(iso, f)


In [106]:
#one-class svm



In [147]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(
    kernel="rbf",
    # ↑ Kernel defines shape of the boundary
    #   rbf = nonlinear, default, most common
    #   linear = simple, high bias
    #   sigmoid = rarely stable

    nu=0.03,
    # ↑ Upper bound on fraction of anomalies
    #   Lower bound on fraction of support vectors
    #   Typical range: 0.01 – 0.10

    gamma="scale",
    # ↑ Controls boundary smoothness
    #   small gamma → smooth, global boundary
    #   large gamma → wiggly, overfitting
    #   scale gamma = 1 / (n_features * variance(X))
    #    auto  gamma = 1 / n_features
    # you can manully control how tight the boundary
    #  0.001   Very smooth, underfits
    #0.01   Smooth
    #1   Very tight

)
ocsvm.fit(X)


OneClassSVM(nu=0.03)

In [148]:
# decision_function:
#   positive → inside boundary (normal)
#   negative → outside (anomaly)

df["score_ocsvm"] = -ocsvm.decision_function(X)

# binary labels
df["anomaly_ocsvm"] = (ocsvm.predict(X) == -1).astype(int)
df.head()

,report_month,report_year,report_quarter,system,users_total,transactions_total,transaction_value_jod,user_jordanian_count,user_non_jordanian_count,user_male_count,...,ach_gbp_transaction_value,returned_cheque_count,returned_cheque_value_jod,returned_cheque_count_rate_pct,returned_cheque_value_rate_pct,activity_level,score_iso,anomaly_iso,score_ocsvm,anomaly_ocsvm
0,2022-11,2022,Q4,CliQ,523000,985000,175000000,494200,24100,342500,...,0,0,0,0.0,0.0,Low Activity,-0.093743,0,-0.050857,0
1,2022-11,2022,Q4,JoMoPay,2020000,1260000,155000000,1760000,250000,1230000,...,0,0,0,0.0,0.0,Low Activity,-0.089450,0,-0.124685,0
2,2022-11,2022,Q4,eFAWATEERcom,3580000,3630000,848000000,0,0,0,...,0,0,0,0.0,0.0,Low Activity,-0.098492,0,-0.136423,0
3,2022-12,2022,Q4,CliQ,561000,1180000,209000000,529800,26300,365000,...,0,0,0,0.0,0.0,Low Activity,-0.106223,0,-0.057087,0
4,2022-12,2022,Q4,JoMoPay,2050000,1510000,188000000,1780000,250000,1230000,...,0,0,0,0.0,0.0,Low Activity,-0.108297,0,-0.031753,0


In [149]:
for g in ["scale", 0.01, 0.1]:
    oc = OneClassSVM(nu=0.03, gamma=g)
    oc.fit(X)

    scores = -oc.decision_function(X_synth)

    print(
        "gamma:", g,
        "roc:", roc_auc_score(df_synth["is_synthetic"], scores)
    )

gamma: scale roc: 0.6802507836990596
gamma: 0.01 roc: 0.5942678011643528
gamma: 0.1 roc: 0.6623376623376624


In [150]:
import joblib

joblib.dump(iso, "prediction_schema_iso.pkl")
joblib.dump(scaler, "prediction_schema_scaler.pkl")
joblib.dump(numerical_features, "prediction_schema_features.pkl")

['prediction_schema_features.pkl']

In [131]:
# evaluation 

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score

X_synth = scaler.transform(df_synth[numerical_features])

iso = IsolationForest(contamination=0.03, n_estimators=300, random_state=42).fit(X)
iso_auc = roc_auc_score(df_synth["is_synthetic"], -iso.decision_function(X_synth))

ocsvm = OneClassSVM(kernel="rbf", nu=0.03, gamma="scale").fit(X)
ocsvm_auc = roc_auc_score(df_synth["is_synthetic"], -ocsvm.decision_function(X_synth))

print(f"IsolationForest AUC: {iso_auc:.3f}")
print(f"OneClassSVM AUC:     {ocsvm_auc:.3f}")

IsolationForest AUC: 0.740
OneClassSVM AUC:     0.675


In [ ]:
import pickle

for filename, artifact in (
    ("prediction_schema_isoforest.pkl", iso),      
    ("prediction_schema_scaler.pkl", scaler),
    ("prediction_schema_features.pkl", numerical_features),
):
    with open(filename, "wb") as f:
        pickle.dump(artifact, f)

print("Saved.")

Saved.
